<a href="https://colab.research.google.com/github/Hemashree-N12/IN226086502_FASTAPI/blob/main/Fine_Tuning_BERT_on_Kaggle_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Objective**

Learn BERT fine-tuning, use transformer pipelines, perform experiments, and evaluate model performance using multiple metrics.

# Step 1:  Data Preprocessing

In [9]:
import pandas as pd, re

data = pd.read_csv("IMDB Dataset.csv", engine='python', on_bad_lines='skip').dropna() #Save the dataset and notebook in same folder for easier access

def preprocess(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)        # strips HTML
    text = re.sub(r"[^a-z\s]", "", text)     # keepa only letters/spaces
    return text.strip()

data['review'] = data['review'].apply(preprocess)

# Step 2: Data Splitting

In [10]:
from sklearn.model_selection import train_test_split

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    data['review'], data['sentiment'], test_size=0.3, random_state=42
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

# Step 3: Tokenization

In [11]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_enc = tokenizer(list(train_texts), truncation=True, padding=True, max_length=256)
val_enc   = tokenizer(list(val_texts),   truncation=True, padding=True, max_length=256)
test_enc  = tokenizer(list(test_texts),  truncation=True, padding=True, max_length=256)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Step 4: Model Building

In [12]:
import torch
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification

class IMDbDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        # Convert sentiment labels into binary (positive=1, negative=0)
        self.labels = [1 if l == "positive" else 0 for l in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Wrap tokenized data into Dataset objects
train_ds = IMDbDataset(train_enc, train_labels)
val_ds   = IMDbDataset(val_enc, val_labels)
test_ds  = IMDbDataset(test_enc, test_labels)

# Load pre-trained BERT model using AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Step 5: Fine Tuning

In [ ]:
from torch.optim import AdamW   # use PyTorch's AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

optimizer = AdamW(model.parameters(), lr=2e-5)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader    = DataLoader(val_ds, batch_size=16)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(1): # Reduced to 1 epoch for faster execution
    model.train()
    print(f"\nEpoch {epoch+1}")
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        optimizer.zero_grad()
        inputs = {k: v.to(device) for k,v in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()


Epoch 1


Training Epoch 1:   0%|          | 0/2188 [00:00<?, ?it/s]

# Step 6: Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm.auto import tqdm

test_loader = DataLoader(test_ds, batch_size=16)
model.eval()
preds, gold = [], []

print("\nStarting Evaluation...")
for batch in tqdm(test_loader, desc="Evaluating Model"):
    inputs = {k: v.to(device) for k,v in batch.items() if k!="labels"}
    labels = batch['labels'].to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
    gold.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(gold, preds))
print("Precision:", precision_score(gold, preds))
print("Recall:", recall_score(gold, preds))
print("F1:", f1_score(gold, preds))
print("Confusion Matrix:\n", confusion_matrix(gold, preds))

# Experiments

### Experiment 1: Freeze all BERT layers (train only the classifier head)

In [ ]:
# Freeze all BERT layers
for param in model.bert.parameters():
    param.requires_grad = False   # classifier head only will train

# Re-initialize optimizer for current trainable parameters
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

print("Starting training for Experiment 1 (frozen BERT layers)...")
for epoch in range(1): # Reduced to 1 epoch for faster execution
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        inputs = {k: v.to(device) for k,v in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

# Evaluation for Experiment 1
model.eval()
preds_exp1, gold_exp1 = [], []
for batch in test_loader:
    inputs = {k: v.to(device) for k,v in batch.items() if k!="labels"}
    labels = batch['labels'].to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    preds_exp1.extend(torch.argmax(logits, dim=1).cpu().numpy())
    gold_exp1.extend(labels.cpu().numpy())

print("\n--- Results for Experiment 1 (Frozen BERT Layers) ---")
print("Accuracy:", accuracy_score(gold_exp1, preds_exp1))
print("Precision:", precision_score(gold_exp1, preds_exp1))
print("Recall:", recall_score(gold_exp1, preds_exp1))
print("F1:", f1_score(gold_exp1, preds_exp1))
print("Confusion Matrix:\n", confusion_matrix(gold_exp1, preds_exp1))

### Experiment 2: Fine-tune last 2 layers of BERT

In [ ]:
# Reset requires_grad for all BERT parameters to False, then selectively unfreeze
for param in model.bert.parameters():
    param.requires_grad = False

# Fine-tune last 2 layers of BERT
for name, param in model.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True   # unfreeze last two layers

# Re-initialize optimizer for current trainable parameters
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

print("\nStarting training for Experiment 2 (fine-tuning last 2 layers)...")
for epoch in range(1): # Reduced to 1 epoch for faster execution
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        inputs = {k: v.to(device) for k,v in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

# Evaluation for Experiment 2
model.eval()
preds_exp2, gold_exp2 = [], []
for batch in test_loader:
    inputs = {k: v.to(device) for k:v in batch.items() if k!="labels"}
    labels = batch['labels'].to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    preds_exp2.extend(torch.argmax(logits, dim=1).cpu().numpy())
    gold_exp2.extend(labels.cpu().numpy())

print("\n--- Results for Experiment 2 (Fine-tune Last 2 Layers) ---")
print("Accuracy:", accuracy_score(gold_exp2, preds_exp2))
print("Precision:", precision_score(gold_exp2, preds_exp2))
print("Recall:", recall_score(gold_exp2, preds_exp2))
print("F1:", f1_score(gold_exp2, preds_exp2))
print("Confusion Matrix:\n", confusion_matrix(gold_exp2, preds_exp2))

### Experiment 3: Unfreeze everything (full fine-tuning)

In [ ]:
# Unfreeze everything
for param in model.bert.parameters():
    param.requires_grad = True

# Re-initialize optimizer for current trainable parameters
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

print("\nStarting training for Experiment 3 (full fine-tuning)...")
for epoch in range(1): # Reduced to 1 epoch for faster execution
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        inputs = {k: v.to(device) for k,v in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

# Evaluation for Experiment 3
model.eval()
preds_exp3, gold_exp3 = [], []
for batch in test_loader:
    inputs = {k: v.to(device) for k:v in batch.items() if k!="labels"}
    labels = batch['labels'].to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    preds_exp3.extend(torch.argmax(logits, dim=1).cpu().numpy())
    gold_exp3.extend(labels.cpu().numpy())

print("\n--- Results for Experiment 3 (Full Fine-tuning) ---")
print("Accuracy:\t", accuracy_score(gold_exp3, preds_exp3))
print("Precision:\t", precision_score(gold_exp3, preds_exp3))
print("Recall:\t\t", recall_score(gold_exp3, preds_exp3))
print("F1:\t\t", f1_score(gold_exp3, preds_exp3))
print("Confusion Matrix:\n", confusion_matrix(gold_exp3, preds_exp3))
print("\nAll experiments completed!")

# Bonus

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Choose model: DistilBERT
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

In [ ]:
from torch.optim import AdamW
from transformers import get_scheduler

optimizer = AdamW(model.parameters(), lr=2e-5)
num_training_steps = len(train_loader) * 3
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

In [ ]:
best_val_loss = float("inf")
patience = 2
patience_counter = 0

for epoch in range(3):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            inputs = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**inputs)
            val_loss += outputs.loss.item()
    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}, Validation Loss: {val_loss:.4f}")

    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered!")
            break

## Conclusion

This notebook demonstrates a complete workflow for fine-tuning a BERT-based model for sentiment analysis. We covered data preprocessing, splitting, tokenization, model building, fine-tuning, and evaluation. Additionally, we explored advanced concepts such as using DistilBERT, implementing a learning rate scheduler, and applying early stopping for more robust model training. The experiments section outlined different strategies for freezing and unfreezing BERT layers, which are crucial for optimizing model performance and resource usage in transfer learning scenarios.